<a href="https://colab.research.google.com/github/matashova12345/Application-Deployment/blob/main/Titanic_KNN_%EC%8B%A4%EC%8A%B5%EA%B3%BC%EC%A0%9C_27_11_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**머신러닝 실습 과제**

목표: Titanic Train Dataset을 전처리하고 새로운 특성을 생성한 뒤, K-Nearest Neighbors(KNN) 분류 모델을 학습합니다. 5-Fold Stratified 교차 검증을 수행하고 GridSearchCV로 최적의 매개변수를 찾습니다.

> 이 노트북은 과제 지시서(PDF)의 요구사항에 따라 작성되었습니다. Google Colab에서 위쪽 셀부터 순서대로 실행하세요. Accuracy 결과는 셀을 실행한 후 표시됩니다.

## 1. 라이브러리 및 데이터 불러오기

실습용 Titanic Train Dataset을 사용합니다. 목표 변수는 `Survived`이며, 1은 생존, 0은 사망을 의미합니다.

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

SEED = 42

# Titanic 학습 데이터셋을 불러옵니다
url = "https://raw.githubusercontent.com/wesm/pydata-book/3rd-edition/datasets/titanic/train.csv"
df = pd.read_csv(url)

print("데이터 크기:", df.shape)
display(df.head())


데이터 크기: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. 데이터 탐색(EDA)

열의 데이터 유형, 기초 통계량 및 결측치 개수를 확인합니다.

In [2]:
display(df.info())
display(df.describe(include="all").T)
missing = pd.DataFrame({
    "Пропусков": df.isna().sum(),
    "Доля, %": (df.isna().mean() * 100).round(2)
})
display(missing.sort_values("Доля, %", ascending=False))
print("목표 변수 Survived의 분포:")
display(df["Survived"].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PassengerId,891.0,NaN,NaN,NaN,446.0,257.353842,1.0,223.5,446.0,668.5,891.0
Survived,891.0,NaN,NaN,NaN,0.383838,0.486592,0.0,0.0,0.0,1.0,1.0
Pclass,891.0,NaN,NaN,NaN,2.308642,0.836071,1.0,2.0,3.0,3.0,3.0
Name,891,891,"Braund, Mr. Owen Harris",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,891,2,male,577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,714.0,NaN,NaN,NaN,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0
SibSp,891.0,NaN,NaN,NaN,0.523008,1.102743,0.0,0.0,0.0,1.0,8.0
Parch,891.0,NaN,NaN,NaN,0.381594,0.806057,0.0,0.0,0.0,0.0,6.0
Ticket,891,681,347082,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,891.0,NaN,NaN,NaN,32.204208,49.693429,0.0,7.9104,14.4542,31.0,512.3292


,Пропусков,"Доля, %"
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22
PassengerId,0,0.00
Name,0,0.00
Pclass,0,0.00
Survived,0,0.00
Sex,0,0.00
Parch,0,0.00
SibSp,0,0.00


목표 변수 Survived의 분포:


,count
Survived,
0,549
1,342


### 주요 열의 의미

- `Survived`: 목표값(0 또는 1).
- `Pclass`: 승객 티켓 등급(1, 2 또는 3).
- `Sex`: 승객의 성별.
- `Age`: 나이.
- `SibSp`: 함께 탑승한 형제자매 및 배우자 수.
- `Parch`: 함께 탑승한 부모 및 자녀 수.
- `Fare`: 티켓 요금.
- `Embarked`: 승선 항구.
- `Cabin`: 객실 번호.

`PassengerId`, `Name`, `Ticket`은 모델의 입력값으로 직접 사용하지 않습니다. 대신 `Name`에서 호칭(`Title`)을 추출해 새로운 특성으로 사용합니다.

## 3. 데이터 전처리 및 특성 공학(Feature Engineering)

중요 사항: 결측치 대체, 인코딩 및 스케일링은 **Pipeline 내부에서 수행**합니다. 따라서 교차 검증 시 각 fold의 학습 데이터만을 사용해 통계값을 계산하며, 데이터 누수(Data Leakage)를 방지할 수 있습니다.

In [3]:
class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = X.copy()
        age_data = X[["Pclass", "Sex", "Age"]].copy()
        self.age_group_medians_ = age_data.groupby(
            ["Pclass", "Sex"]
        )["Age"].median()
        self.age_median_ = X["Age"].median()
        return self

    def transform(self, X):
        X = X.copy()

        # Age: Pclass와 Sex 그룹별 중앙값 사용; 해당 그룹이 없으면 학습 데이터 전체 중앙값 사용
        group_keys = list(zip(X["Pclass"], X["Sex"]))
        group_medians = [
            self.age_group_medians_.get(key, self.age_median_)
            for key in group_keys
        ]
        X["Age"] = X["Age"].fillna(pd.Series(group_medians, index=X.index))

        # 새로운 특성 생성
        X["Family_Size"] = X["SibSp"] + X["Parch"] + 1

        def family_group(size):
            if size == 1:
                return "Alone"
            elif size <= 3:
                return "Small"
            elif size <= 5:
                return "Medium"
            return "Large"

        X["Family_Size_Grouped"] = X["Family_Size"].apply(family_group)

        # 이름에서 호칭 추출: 예) Mr, Mrs, Miss, Master
        X["Title"] = X["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
        common_titles = ["Mr", "Mrs", "Miss", "Master"]
        X["Title"] = X["Title"].where(X["Title"].isin(common_titles), "Rare")

        # 객실 정보의 결측 여부 특성
        X["Cabin_missing"] = X["Cabin"].isna().astype(int)

        # 원본 식별자 및 사용하지 않는 텍스트 열 제거
        X = X.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

        return X


### 3.1. X와 y 분리

`X`는 승객의 입력 특성이고, `y`는 정답인 `Survived`입니다.

In [4]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

print("X 데이터 크기:", X.shape)
print("y 데이터 크기:", y.shape)
display(X.head())


X 데이터 크기: (891, 11)
y 데이터 크기: (891,)


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### 3.2. 인코딩 및 스케일링

- 범주형 특성(`Sex`, `Embarked`, `Pclass`, `Title`, 가족 규모)은 One-Hot Encoding으로 변환합니다.
- 수치형 특성의 결측치는 중앙값으로 대체하고 StandardScaler로 스케일링합니다.
- `Embarked`의 결측치는 최빈값으로 대체합니다.
- `Fare`의 결측치는 중앙값으로 대체합니다.
- `Cabin`은 결측 여부를 나타내는 `Cabin_missing` 특성을 생성한 후 원래 열을 제거합니다.

In [5]:
numeric_features = [
    "Age", "SibSp", "Parch", "Fare", "Family_Size", "Cabin_missing"
]
categorical_features = [
    "Pclass", "Sex", "Embarked", "Family_Size_Grouped", "Title"
]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# 전체 Pipeline: 특성 생성 -> 전처리 -> KNN 모델
knn_pipeline = Pipeline(steps=[
    ("features", TitanicFeatureEngineer()),
    ("preprocessing", preprocessor),
    ("model", KNeighborsClassifier())
])

print("Pipeline подготовлен.")


Pipeline подготовлен.


## 4. KNN 모델 학습 및 5-Fold Stratified 교차 검증

`StratifiedKFold`는 데이터를 5개 부분(fold)으로 나누며, 각 fold에서 생존자와 사망자의 비율이 비슷하게 유지되도록 합니다.

먼저 기본 설정의 KNN 모델을 평가하고 Accuracy를 확인합니다.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

baseline_scores = cross_val_score(
    knn_pipeline,
    X,
    y,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("각 fold의 Accuracy:", np.round(baseline_scores, 4))
print(f"튜닝 전 평균 Accuracy: {baseline_scores.mean():.4f}")
print(f"표준편차: {baseline_scores.std():.4f}")


각 fold의 Accuracy: [0.8212 0.8202 0.7753 0.8202 0.8202]
튜닝 전 평균 Accuracy: 0.8114
표준편차: 0.0181


## 5. GridSearchCV를 이용한 하이퍼파라미터 튜닝

과제에서 제시한 다음 매개변수를 탐색합니다:

- `n_neighbors`: 이웃의 수(3, 5, 7, 9, 11, 15).
- `weights`: 이웃에 동일한 가중치를 주는 `uniform` 또는 거리에 따라 가중치를 주는 `distance`.
- `p`: 1(맨해튼 거리) 또는 2(유클리드 거리).

전체 Pipeline을 GridSearchCV에 전달하므로 각 fold 안에서 전처리 과정도 별도로 학습됩니다.

In [7]:
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    refit=True
)

grid_search.fit(X, y)

print("최적 매개변수:")
print(grid_search.best_params_)
print(f"최적 교차 검증 평균 Accuracy: {grid_search.best_score_:.4f}")


최적 매개변수:
{'model__n_neighbors': 7, 'model__p': 1, 'model__weights': 'uniform'}
최적 교차 검증 평균 Accuracy: 0.8249


## 6. 튜닝 전후 Accuracy 비교

In [8]:
comparison = pd.DataFrame({
    "모델": ["튜닝 전 KNN", "GridSearchCV 튜닝 후 KNN"],
    "교차 검증 평균 Accuracy": [
        baseline_scores.mean(),
        grid_search.best_score_
    ]
})
comparison["교차 검증 평균 Accuracy"] = comparison["교차 검증 평균 Accuracy"].round(4)
display(comparison)

improvement = grid_search.best_score_ - baseline_scores.mean()
print(f"Accuracy 변화량: {improvement:+.4f}")


,모델,교차 검증 평균 Accuracy
0,튜닝 전 KNN,0.8114
1,GridSearchCV 튜닝 후 KNN,0.8249


Accuracy 변화량: +0.0135


## 7. 최종 결과 해석

1. Titanic 데이터셋의 결측치를 처리하고 Family_Size, Family_Size_Grouped, Title 등의 새로운 특성을 생성하였다.
2. 범주형 데이터는 One-Hot Encoding을 적용하고, 수치형 데이터는 StandardScaler를 사용하여 스케일링하였다.
3. KNN 모델의 튜닝 전 교차 검증 평균 Accuracy는 0.8114(81.14%)로 나타났다.
4. GridSearchCV를 통해 최적의 매개변수(n_neighbors=7, weights='uniform', p=1)를 찾았으며, 튜닝 후 Accuracy는 0.8249(82.49%)로 향상되었다.
5. 따라서 하이퍼파라미터 튜닝을 통해 모델의 Accuracy가 0.0135(1.35%p) 향상되었다.